# SentimentLab AI (P4): Hugging Face BERT Fine-Tuning
### Academic Mini-Project: Generative vs Discriminative Sentiment Analysis Laboratory

This Google Colab notebook provides an end-to-end, reproducible pipeline to fine-tune **`bert-base-uncased`** on an NVIDIA T4 GPU for 3-class sentiment analysis (`0=Negative`, `1=Neutral`, `2=Positive`).

**Hardware Recommendation:**
- Colab Menu: `Runtime` -> `Change runtime type` -> Hardware accelerator: `T4 GPU`.

In [ ]:
# Step 1: Verify GPU
import torch
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device Name: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")
else:
    print("WARNING: Running on CPU. Please switch to T4 GPU runtime!")

In [ ]:
# Step 2: Install Required Libraries
!pip install -q transformers datasets accelerate scikit-learn pandas matplotlib seaborn pyarrow

In [ ]:
# Step 3: Load or Synthesize Preprocessed Yelp Dataset
import os
import numpy as np
import pandas as pd

# If uploading directly from local project artifacts:
# from google.colab import files
# uploaded = files.upload() # upload train_real.parquet, val_real.parquet, eval_subset_real.parquet

# Otherwise, build directly from Hugging Face Yelp Review Full:
from datasets import load_dataset

print("Loading Yelp Review Full dataset from Hugging Face...")
dataset = load_dataset("yelp_review_full")

def map_stars_to_3class(example):
    # Yelp stars: 0, 1 -> 0 (Negative); 2 -> 1 (Neutral); 3, 4 -> 2 (Positive)
    label = example["label"]
    if label in (0, 1):
        example["sentiment_label"] = 0
    elif label == 2:
        example["sentiment_label"] = 1
    else:
        example["sentiment_label"] = 2
    return example

dataset = dataset.map(map_stars_to_3class)

# Sample 19,200 train (6,400 per class) and 4,800 val (1,600 per class)
train_df = pd.DataFrame(dataset["train"])
test_df = pd.DataFrame(dataset["test"])

def stratified_sample(df, n_per_class, seed=42):
    return df.groupby("sentiment_label", group_keys=False).apply(
        lambda x: x.sample(n=n_per_class, random_state=seed)
    ).sample(frac=1.0, random_state=seed).reset_index(drop=True)

train_subset = stratified_sample(train_df, 6400, seed=42)
# Remaining train pool for validation:
rem_train = train_df.drop(train_subset.index, errors="ignore")
val_subset = stratified_sample(rem_train, 1600, seed=42)
eval_subset = stratified_sample(test_df, 1000, seed=42)

print(f"Splits ready: Train={len(train_subset)}, Val={len(val_subset)}, Eval={len(eval_subset)}")

In [ ]:
# Step 4: Tokenization & PyTorch Datasets
from transformers import AutoTokenizer
import torch
from torch.utils.data import Dataset, DataLoader

MODEL_NAME = "bert-base-uncased"
MAX_LEN = 128
BATCH_SIZE = 16

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

class SentimentDataset(Dataset):
    def __init__(self, texts, labels):
        self.encodings = tokenizer(texts, truncation=True, padding="max_length", max_length=MAX_LEN, return_tensors="pt")
        self.labels = torch.tensor(labels, dtype=torch.long)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        item = {k: v[idx] for k, v in self.encodings.items()}
        item["labels"] = self.labels[idx]
        return item

train_ds = SentimentDataset(train_subset["text"].tolist(), train_subset["sentiment_label"].tolist())
val_ds = SentimentDataset(val_subset["text"].tolist(), val_subset["sentiment_label"].tolist())
eval_ds = SentimentDataset(eval_subset["text"].tolist(), eval_subset["sentiment_label"].tolist())

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False)
eval_loader = DataLoader(eval_ds, batch_size=BATCH_SIZE, shuffle=False)
print("Dataloaders initialized.")

In [ ]:
# Step 5: Fine-Tune BERT with AdamW and Early Stopping
from transformers import AutoModelForSequenceClassification, get_linear_schedule_with_warmup
from torch.optim import AdamW
from sklearn.metrics import accuracy_score, f1_score
import time

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=3)
model.to(device)

EPOCHS = 3
LR = 2e-5
optimizer = AdamW(model.parameters(), lr=LR, weight_decay=0.01)
total_steps = len(train_loader) * EPOCHS
scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=int(0.1*total_steps), num_training_steps=total_steps)

history = {"train_loss": [], "val_loss": [], "val_macro_f1": []}
best_val_loss = float("inf")
os.makedirs("bert_checkpoint", exist_ok=True)

print(f"Starting BERT Fine-Tuning on {device} for {EPOCHS} epochs...")
for epoch in range(1, EPOCHS + 1):
    model.train()
    total_loss = 0.0
    for step, batch in enumerate(train_loader):
        optimizer.zero_grad()
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        total_loss += loss.item()

    avg_train_loss = total_loss / len(train_loader)

    # Validation
    model.eval()
    val_loss = 0.0
    preds, targets = [], []
    with torch.no_grad():
        for batch in val_loader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)
            outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
            val_loss += outputs.loss.item()
            p = torch.argmax(outputs.logits, dim=-1).cpu().numpy()
            preds.extend(p)
            targets.extend(labels.cpu().numpy())

    avg_val_loss = val_loss / len(val_loader)
    macro_f1 = f1_score(targets, preds, average="macro")
    history["train_loss"].append(avg_train_loss)
    history["val_loss"].append(avg_val_loss)
    history["val_macro_f1"].append(macro_f1)

    print(f"Epoch {epoch}/{EPOCHS} - Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f} | Val Macro-F1: {macro_f1:.4f}")
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        model.save_pretrained("bert_checkpoint")
        tokenizer.save_pretrained("bert_checkpoint")
        print(f"Saved best model checkpoint (val_loss: {avg_val_loss:.4f})")

In [ ]:
# Step 6: Final Evaluation on Fixed 3,000 Evaluation Subset
import json
from sklearn.metrics import classification_report, confusion_matrix

model = AutoModelForSequenceClassification.from_pretrained("bert_checkpoint").to(device)
model.eval()

eval_preds, eval_probs, eval_targets = [], [], []
start_eval = time.time()
with torch.no_grad():
    for batch in eval_loader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        probs = torch.softmax(outputs.logits, dim=-1).cpu().numpy()
        p = np.argmax(probs, axis=1)
        eval_probs.extend(probs)
        eval_preds.extend(p)
        eval_targets.extend(labels.cpu().numpy())

eval_duration = time.time() - start_eval
acc = accuracy_score(eval_targets, eval_preds)
f1 = f1_score(eval_targets, eval_preds, average="macro")
cm = confusion_matrix(eval_targets, eval_preds).tolist()

eval_metrics = {
    "accuracy": float(acc),
    "macro_f1": float(f1),
    "confusion_matrix": cm,
    "inference_latency_per_sample_ms": float((eval_duration / len(eval_subset)) * 1000.0),
}
with open("bert_checkpoint/evaluation_metrics.json", "w") as f:
    json.dump(eval_metrics, f, indent=2)
with open("bert_checkpoint/training_history.json", "w") as f:
    json.dump(history, f, indent=2)

print(f"BERT Evaluation Subset Accuracy: {acc:.4f} | Macro-F1: {f1:.4f}")

In [ ]:
# Step 7: Export Artifacts Archive for SentimentLab AI
!zip -r bert_artifacts.zip bert_checkpoint/
from google.colab import files
files.download("bert_artifacts.zip")
print("Downloaded bert_artifacts.zip. Extract contents into artifacts/models/bert/ in your project directory!")